# MIDI Generation Plugin - From-Scratch Training (Kaggle T4)

End-to-end pipeline that produces `model_best.ts.pt` + `vocab.json` for the JUCE plugin.

**Required Kaggle setup**
- GPU: T4 x1 (or P100); accelerator must be ON before running
- Internet: ON (for pip installs)
- Add input dataset with raw MIDI files; the notebook will symlink it into `dataset/midi_raw/`

**Steps**
1. Clone repo and install deps
2. Preprocess raw MIDI -> meta JSONL
3. Tokenize -> Performance tokens (with `<GENRE_TRAP>` + `<KEY_*>` prefix)
4. Build vocab.json + split + chunk
5. Train Transformer LM (~25M params, fp16 AMP, ~2-4 h on T4)
6. Export to TorchScript and download

The output artifacts are written to `/kaggle/working/checkpoints/` and copied into
`/kaggle/working/<repo>/plugin/juce/bin/` ready to be downloaded and dropped into
the JUCE plugin folder. **Nothing in the C++ plugin needs to change.**

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import torch

print("python:", sys.version.split()[0])
print("torch :", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("vram :", round(torch.cuda.get_device_properties(0).total_memory / 1024 ** 3, 2), "GB")

## 1. Get the code

Two options - either clone the repo from GitHub, or upload the `dataset/` and `model/`
folders as a Kaggle Dataset and unpack here. Edit the cell below for your case.

In [ ]:
REPO_URL = "https://github.com/Dizzers/MIDI-Generation-Plugin.git"
WORK_DIR = Path("/kaggle/working")
REPO_DIR = WORK_DIR / "MIDI-Generation-Plugin"
PROJECT_DIR = REPO_DIR / "DIPLOM SPACE"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

os.chdir(PROJECT_DIR)
print("cwd:", os.getcwd())
print("contents:", sorted(os.listdir()))

In [ ]:
!pip install -q "music21>=9.0" "tqdm>=4.66" "mido>=1.3" "numpy>=1.26" "matplotlib>=3.8"

## 2. Mount the MIDI dataset

Add your raw-MIDI Kaggle dataset (e.g. `/kaggle/input/my-trap-midis/`) and update
`RAW_DATASET_DIR` below. We symlink it into `dataset/midi_raw/` to keep paths uniform.

In [ ]:
RAW_DATASET_DIR = Path("/kaggle/input/your-midi-dataset")  # <-- edit me

raw_target = PROJECT_DIR / "dataset" / "midi_raw"
raw_target.parent.mkdir(parents=True, exist_ok=True)

if raw_target.exists() and raw_target.is_symlink():
    raw_target.unlink()
elif raw_target.exists():
    import shutil
    shutil.rmtree(raw_target)

raw_target.symlink_to(RAW_DATASET_DIR)
sample = list(raw_target.rglob("*.mid"))[:5] + list(raw_target.rglob("*.midi"))[:5]
print(f"linked {RAW_DATASET_DIR} -> {raw_target}")
print("first MIDI files:")
for p in sample:
    print(" ", p)

## 3. Build the dataset
Preprocess -> tokenize -> vocab -> split -> chunk. Each step writes JSON stats so you
can sanity-check throughput and key distribution before training.

In [ ]:
!python -m dataset.preprocess_midi

In [ ]:
!python -m dataset.tokenize_midi
!python -m dataset.build_vocab
!python -m dataset.split_tokens
!python -m dataset.chunk_tokens

## 4. Train

T4-friendly defaults: `batch_size=16`, `grad_accum_steps=2`, fp16 AMP, cosine LR with
4-epoch warmup, early-stop after 10 epochs without val improvement.

Adjust `--num_epochs` lower for fast smoke tests.

In [ ]:
!python -m model.train \
    --num_epochs 60 \
    --batch_size 16 \
    --grad_accum_steps 2 \
    --learning_rate 3e-4 \
    --max_len 1024 \
    --d_model 512 \
    --n_layers 8 \
    --d_ff 2048 \
    --dropout 0.2 \
    --num_workers 2 \
    --device cuda

## 5. Export to TorchScript and copy to plugin/juce/bin/

The `--copy-to-bin` flag drops `model_best.ts.pt` and `vocab.json` straight into the
JUCE plugin bin directory. After this you can simply download those two files from
`/kaggle/working/` and replace the matching files in your local plugin tree.

In [ ]:
!python -m model.export_torchscript --device cpu --copy-to-bin
!ls -la "plugin/juce/bin/"

## 6. Sample generation (optional)

Quick smoke-test that the model produces something musical at all. The .mid is saved
to `generated/sample.mid` and can be downloaded from the Kaggle file browser.

In [ ]:
!python -m model.generate \
    --key C_MAJOR --seed 42 \
    --temperature 0.95 --top_k 12 --top_p 0.9 \
    --target_seconds 8.0 --bpm 140 \
    --out generated/sample_C_MAJOR.mid

!python -m model.generate \
    --key A_MINOR --seed 7 \
    --temperature 1.05 --top_k 24 --top_p 0.92 \
    --target_seconds 8.0 --bpm 90 \
    --out generated/sample_A_MINOR.mid

!ls -la generated/